# Week 1 — Pilot Run（小样本测试）

**目标**：用 ~20 家公司 × 3年 把整条 pipeline 跑通，确认：
1. EDGAR API 能正常拉 filing list
2. MD&A 提取逻辑正确
3. WRDS Compustat / CRSP 能匹配上
4. 最终 `master_panel` 结构正确

跑通后把 `PILOT_TICKERS` 换成全量 S&P 1500 即可。

## 0. 配置

In [1]:
# !pip install wrds pandas pyarrow tqdm requests beautifulsoup4 lxml

import re, time, json, requests, warnings
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm
from bs4 import BeautifulSoup

warnings.filterwarnings('ignore')

# ── 唯一需要改的地方 ────────────────────────────────────────────────────────
SEC_USER_AGENT = "YourName YourEmail@uchicago.edu"   # ← 改成你自己的
WRDS_USERNAME  = "your_wrds_username"                # ← 改成你的 WRDS 账号

# Pilot：用 20 家覆盖多行业的大公司（ticker 要能在 EDGAR 查到）
PILOT_TICKERS = [
    "AAPL", "MSFT", "GOOGL", "AMZN",   # Tech
    "JPM",  "BAC",  "GS",               # Finance
    "JNJ",  "PFE",                       # Healthcare
    "XOM",  "CVX",                       # Energy
    "WMT",  "HD",                        # Retail
    "BA",   "GE",                        # Industrials
    "VZ",   "T",                         # Telecom
    "NEE",                               # Utilities
    "CAT",  "MMM",                       # Manufacturing
]

# Pilot：只拉 3 年
PILOT_START = 2018
PILOT_END   = 2020

DATA_DIR    = Path("data_pilot")
RAW_DIR     = DATA_DIR / "raw"
FILINGS_DIR = DATA_DIR / "filings"
OUTPUT_DIR  = DATA_DIR / "processed"
for d in [RAW_DIR, FILINGS_DIR, OUTPUT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

HEADERS    = {"User-Agent": SEC_USER_AGENT}
RATE_SLEEP = 0.15   # SEC 限速：~7 req/s，保守

print(f"Pilot 公司: {len(PILOT_TICKERS)} 家，{PILOT_START}–{PILOT_END}")

Pilot 公司: 20 家，2018–2020


/opt/anaconda3/envs/111/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


---
## Step 1: 从 SEC JSON 获取 ticker → CIK 对照

In [2]:
# 下载 SEC 全量 ticker-CIK 映射（~3MB JSON，几秒搞定）
r = requests.get(
    "https://www.sec.gov/files/company_tickers.json",
    headers=HEADERS, timeout=30
)
r.raise_for_status()

ticker_cik = pd.DataFrame(r.json()).T
ticker_cik.columns = ["cik", "ticker", "company_name"]
ticker_cik["ticker"] = ticker_cik["ticker"].str.upper()
ticker_cik["cik"]    = ticker_cik["cik"].astype(str).str.zfill(10)

# 只保留 pilot 公司
pilot_meta = ticker_cik[ticker_cik["ticker"].isin(PILOT_TICKERS)].copy()
pilot_meta = pilot_meta.drop_duplicates(subset="ticker")

missing = set(PILOT_TICKERS) - set(pilot_meta["ticker"])
if missing:
    print(f"⚠️  以下 ticker 在 EDGAR 找不到: {missing}")

print(f"✅ 匹配到 {len(pilot_meta)} 家公司")
pilot_meta

✅ 匹配到 20 家公司


,cik,ticker,company_name
1,0001652044,GOOGL,Alphabet Inc.
2,0000320193,AAPL,Apple Inc.
3,0000789019,MSFT,MICROSOFT CORP
4,0001018724,AMZN,AMAZON COM INC
9,0000104169,WMT,Walmart Inc.
10,0000019617,JPM,JPMORGAN CHASE & CO
13,0000034088,XOM,EXXON MOBIL CORP
16,0000200406,JNJ,JOHNSON & JOHNSON
23,0000093410,CVX,CHEVRON CORP
24,0000018230,CAT,CATERPILLAR INC


---
## Step 2: 从 EDGAR Submissions API 拉 10-K filing 列表

In [3]:

def get_all_company_filings(cik: str, pause: float = 0.2) -> pd.DataFrame:
    """
    读取 SEC submissions API 的完整 filing history。
    关键修正：不能只读 filings['recent']，还要读取 filings['files'] 里的历史 JSON；
    否则 2018/2019 这类较早 10-K 很容易漏掉。
    """
    cik = str(cik).zfill(10)
    url = f"https://data.sec.gov/submissions/CIK{cik}.json"
    r = requests.get(url, headers=HEADERS, timeout=30)
    r.raise_for_status()
    data = r.json()
    time.sleep(RATE_SLEEP)

    dfs = []

    # 1) recent filings
    recent = pd.DataFrame(data.get("filings", {}).get("recent", {}))
    if len(recent) > 0:
        dfs.append(recent)

    # 2) historical filings stored in extra JSON files
    for f in data.get("filings", {}).get("files", []):
        hist_url = f"https://data.sec.gov/submissions/{f['name']}"
        rr = requests.get(hist_url, headers=HEADERS, timeout=30)
        rr.raise_for_status()
        hist = pd.DataFrame(rr.json())
        if len(hist) > 0:
            dfs.append(hist)
        time.sleep(pause)

    if not dfs:
        return pd.DataFrame()

    out = pd.concat(dfs, ignore_index=True)
    if "accessionNumber" in out.columns:
        out = out.drop_duplicates(subset=["accessionNumber"])

    out["cik"] = cik
    out["company_name"] = data.get("name", "")
    out["ticker_edgar"] = (data.get("tickers") or [None])[0]
    out["sic_edgar"] = data.get("sic", None)
    return out


def get_10k_filings_for_cik(cik: str, start_year: int, end_year: int) -> pd.DataFrame:
    """
    返回指定 fiscal/report year 范围内的 10-K 记录。
    样本年份使用 reportDate 的年份；event date 仍然使用 filingDate。
    """
    df = get_all_company_filings(cik)
    if df.empty:
        return pd.DataFrame()

    df = df[df["form"].isin(["10-K", "10-K405"])].copy()
    df["date_filed"] = pd.to_datetime(df["filingDate"], errors="coerce")
    df["period_of_report"] = pd.to_datetime(df["reportDate"], errors="coerce")
    df["filing_year"] = df["date_filed"].dt.year
    df["report_year"] = df["period_of_report"].dt.year

    # 用 fiscal/report year 定义 firm-year；如果 reportDate 缺失，才 fallback 到 filing year
    df["year"] = df["report_year"].fillna(df["filing_year"]).astype("Int64")
    df = df[df["year"].between(start_year, end_year)].copy()

    df = df.rename(columns={
        "form": "form_type",
        "accessionNumber": "accession_no",
        "primaryDocument": "primary_doc",
    })

    keep_cols = [
        "cik", "company_name", "ticker_edgar", "sic_edgar",
        "form_type", "date_filed", "period_of_report",
        "filing_year", "report_year", "year",
        "accession_no", "primary_doc"
    ]
    return df[keep_cols].sort_values(["year", "date_filed"])


# 遍历 pilot 公司
all_filings = []
for _, row in tqdm(pilot_meta.iterrows(), total=len(pilot_meta), desc="EDGAR submissions"):
    try:
        df = get_10k_filings_for_cik(row["cik"], PILOT_START, PILOT_END)
        if not df.empty:
            df["ticker"] = row["ticker"]
            all_filings.append(df)
    except Exception as e:
        print(f"  ❌ {row['ticker']} ({row['cik']}): {e}")

filings_df = pd.concat(all_filings, ignore_index=True) if all_filings else pd.DataFrame()
filings_df.to_parquet(RAW_DIR / "filings_pilot.parquet")

print(f"\n✅ 获取到 {len(filings_df)} 份 10-K filing 记录")
if len(filings_df) > 0:
    print(filings_df.groupby("ticker")["accession_no"].count().to_string())


EDGAR submissions: 100%|██████████| 20/20 [01:41<00:00,  5.09s/it]


✅ 获取到 59 份 10-K filing 记录
ticker
AAPL     3
AMZN     3
BA       3
BAC      3
CAT      3
CVX      3
GE       3
GOOGL    3
GS       3
HD       3
JNJ      2
JPM      3
MMM      3
MSFT     3
NEE      3
PFE      3
T        3
VZ       3
WMT      3
XOM      3


In [4]:

# 检查：每家公司每个 fiscal/report year 应恰好有 1 份 10-K
cross_check = filings_df.groupby(["ticker", "year"]).size().unstack(fill_value=0)
print("每家公司 × 每年 10-K 数量（应该全是 1）：")
print(cross_check)

expected_grid = pd.MultiIndex.from_product(
    [PILOT_TICKERS, range(PILOT_START, PILOT_END + 1)],
    names=["ticker", "year"]
).to_frame(index=False)
actual_grid = filings_df[["ticker", "year"]].drop_duplicates()
missing_filings = expected_grid.merge(actual_grid, on=["ticker", "year"], how="left", indicator=True)
missing_filings = missing_filings[missing_filings["_merge"] == "left_only"].drop(columns="_merge")

if len(missing_filings) == 0:
    print("\n✅ 10-K filing metadata 覆盖完整。")
else:
    print("\n⚠️  缺失的 ticker-year：")
    print(missing_filings.to_string(index=False))


每家公司 × 每年 10-K 数量（应该全是 1）：
year    2018  2019  2020
ticker                  
AAPL       1     1     1
AMZN       1     1     1
BA         1     1     1
BAC        1     1     1
CAT        1     1     1
CVX        1     1     1
GE         1     1     1
GOOGL      1     1     1
GS         1     1     1
HD         1     1     1
JNJ        1     1     0
JPM        1     1     1
MMM        1     1     1
MSFT       1     1     1
NEE        1     1     1
PFE        1     1     1
T          1     1     1
VZ         1     1     1
WMT        1     1     1
XOM        1     1     1

⚠️  缺失的 ticker-year：
ticker  year
   JNJ  2020


---
## Step 3: 下载 MD&A 文本

In [5]:
BASE_ARCHIVE = "https://www.sec.gov/Archives/edgar/data"

def fetch_text(url: str, retries: int = 3) -> str | None:
    for attempt in range(retries):
        try:
            r = requests.get(url, headers=HEADERS, timeout=30)
            if r.status_code == 200:
                time.sleep(RATE_SLEEP)
                return r.text
            elif r.status_code == 404:
                return None
        except requests.RequestException:
            time.sleep(2 ** attempt)
    return None


def clean_html_to_text(raw_text: str) -> str:
    soup = BeautifulSoup(raw_text, "lxml")
    for tag in soup(["script", "style", "head"]):
        tag.decompose()
    for tag in soup.find_all(re.compile(r"^ix:", re.I)):
        if tag.name and tag.name.lower() in ("ix:header", "ix:hidden", "ix:references"):
            tag.decompose()
    text = soup.get_text(separator=" ", strip=True)
    text = text.replace("\u00a0", " ")   # non-breaking space
    text = text.replace("\u2019", "'")   # right single quotation mark
    text = text.replace("\u2018", "'")   # left  single quotation mark
    text = text.replace("\u201c", '"')
    text = text.replace("\u201d", '"')
    text = re.sub(r"[\t\r\n\f\v]+", " ", text)
    text = re.sub(r" {2,}", " ", text)
    return text


def find_mda_start(tl: str, n: int):
    """
    三层降级策略定位 Item 7 MD&A 起始位置。
    Level 1/2 精确，不设上限（精确模式不会误命中）。
    Level 3 兜底，设 85% 上限防末尾乱码。
    """
    min_pos = max(200, int(n * 0.01))  # 跳过最开头 1%

    # Level 1: item 7 + management...discussion (无上限约束)
    pat_exact = r"item\s+7[.]?\s*management[\s']{0,3}s?\s+discussion"
    cands = [m for m in re.finditer(pat_exact, tl) if m.start() > min_pos]
    if cands:
        return cands[-1].start()

    # Level 2: item 7 后 120 chars 内含 management (无上限约束)
    all_item7 = list(re.finditer(r"item\s+7\b", tl))
    for m in reversed(all_item7):
        pos = m.start()
        if pos <= min_pos:
            continue
        if re.search(r"management", tl[pos: pos + 120]):
            return pos

    # Level 3: 宽松兜底，限 85% 以内避免末尾误命中
    max_pos_fallback = int(n * 0.85)
    cands3 = [m for m in all_item7
              if min_pos < m.start() < max_pos_fallback]
    if cands3:
        return cands3[-1].start()

    return None


ITEM7_END_PATS = [
    r"item\s+7a[.]?\s",
    r"item\s+7a\b",
    r"item\s+8[.]?\s",
    r"quantitative\s+and\s+qualitative\s+disclosures?\s+about\s+market",
]


def extract_mda(raw_text: str, max_chars: int = 120_000) -> str:
    is_html = bool(re.search(r"<html|<htm|<body", raw_text[:500], re.IGNORECASE))
    text = clean_html_to_text(raw_text) if is_html else raw_text
    tl   = text.lower()
    n    = len(text)
    start = find_mda_start(tl, n)
    if start is None:
        return ""
    search_from = start + 200
    end = None
    for pat in ITEM7_END_PATS:
        m = re.search(pat, tl[search_from:])
        if m and m.start() > 300:
            end = search_from + m.start()
            break
    if end is None or (end - start) < 300:
        end = start + max_chars
    return text[start:end].strip()[:max_chars]


print("\u2705 extract_mda 定义完成\n")

# Quick self-test
PAD = "A" * 40000
_t1 = PAD + "Item 7 . 42 "*30 + PAD + " ITEM 7. MANAGEMENT\u2019S DISCUSSION Revenue up. ITEM 7A." + "Z"*5000
_r1 = extract_mda(_t1)
print(f"[T1 iXBRL 卷引号]  {len(_r1):>6,} chars  {'\u2705' if len(_r1)>50 else '\u274c'}")

_t2 = "A"*80000 + "Item 7 . 42 "*10 + " ITEM 7. MANAGEMENT'S DISCUSSION Net sales grew. ITEM 7A." + "Z"*2000
_r2 = extract_mda(_t2)
print(f"[T2 Item7 in 95%]  {len(_r2):>6,} chars  {'\u2705' if len(_r2)>50 else '\u274c'}")

_r3 = extract_mda("A"*5000 + " no mda here " * 100)

print(f"[T3 no Item7]      {len(_r3):>6,} chars  {'✅' if _r3 == '' else '❌'} (empty correct)")

✅ extract_mda 定义完成

[T1 iXBRL 卷引号]   5,052 chars  ✅
[T2 Item7 in 95%]   2,056 chars  ✅
[T3 no Item7]           0 chars  ✅ (empty correct)


In [6]:

# 批量下载（pilot 样本约 60 份，几分钟内完成）
# 注意：year 使用 report_year/fiscal year；date_filed 才是事件研究里的 event date。
mda_records = []

for _, row in tqdm(filings_df.iterrows(), total=len(filings_df), desc="下载 MD&A"):
    archive_cik = str(row["cik"]).lstrip("0")   # EDGAR archive 路径用不带前导零的 CIK
    acc_no      = row["accession_no"]
    pri_doc     = str(row["primary_doc"]) if pd.notna(row["primary_doc"]) else ""
    ticker      = row["ticker"]
    year        = int(row["year"])

    save_dir  = FILINGS_DIR / ticker / str(year)
    save_dir.mkdir(parents=True, exist_ok=True)
    save_path = save_dir / f"{acc_no.replace('-','')}.txt"

    rec = dict(
        cik=row["cik"], ticker=ticker, accession_no=acc_no,
        date_filed=row["date_filed"], year=year,
        mda_path=str(save_path), mda_len=0, status="pending"
    )

    # 断点：文件已存在则跳过，但仍视为成功记录
    if save_path.exists() and save_path.stat().st_size > 200:
        rec["mda_len"] = save_path.stat().st_size
        rec["status"]  = "cached"
        mda_records.append(rec)
        continue

    # --- 构造下载 URL ---
    acc_no_clean = acc_no.replace("-", "")

    # 先尝试 primary_doc
    raw_text = None
    if pri_doc:
        url = f"{BASE_ARCHIVE}/{archive_cik}/{acc_no_clean}/{pri_doc}"
        raw_text = fetch_text(url)

    # 如果失败，查 index 页面找 .htm 文件
    if raw_text is None:
        idx_url  = f"{BASE_ARCHIVE}/{archive_cik}/{acc_no_clean}/{acc_no}-index.htm"
        idx_text = fetch_text(idx_url)
        if idx_text:
            htm_links = re.findall(
                r'href="(/Archives/edgar/data/[^"]+\.htm[^"]*)"|href="([^"]+\.htm[^"]*)"',
                idx_text, re.IGNORECASE
            )
            # 取第一个 .htm（通常是主文件）
            for g1, g2 in htm_links:
                link = g1 or g2
                if link and "index" not in link.lower():
                    full_url = f"https://www.sec.gov{link}" if link.startswith("/") else link
                    raw_text = fetch_text(full_url)
                    if raw_text:
                        break

    if raw_text is None:
        rec["status"] = "error_download"
        print(f"  ❌ 下载失败: {ticker} {year}")
        mda_records.append(rec)
        continue

    # --- 提取 MD&A ---
    mda = extract_mda(raw_text)

    if len(mda) < 300:
        # pilot 阶段不要因为个别公司 Item 7 regex 失败而丢整条 observation；
        # 先 fallback 到清洗后的全文，后续可对这些样本单独检查。
        fallback = clean_html_to_text(raw_text)
        if len(fallback) > 1000:
            mda = fallback[:120_000]
            rec["status"] = "ok_fulltext_fallback"
            print(f"  ⚠️  MD&A 太短，使用全文 fallback: {ticker} {year}")
        else:
            rec["status"] = "error_mda_too_short"
            print(f"  ⚠️  MD&A 太短 ({len(mda)} chars): {ticker} {year}")
            mda_records.append(rec)
            continue
    else:
        rec["status"] = "ok"

    save_path.write_text(mda, encoding="utf-8")
    rec["mda_len"] = len(mda)
    mda_records.append(rec)

mda_meta = pd.DataFrame(mda_records)
mda_meta.to_parquet(RAW_DIR / "mda_meta_pilot.parquet")

print("\n✅ MD&A/文本下载完成")
print(mda_meta["status"].value_counts(dropna=False))


下载 MD&A: 100%|██████████| 59/59 [01:21<00:00,  1.39s/it]


✅ MD&A/文本下载完成
status
ok        52
cached     7
Name: count, dtype: int64


In [7]:
# 抽查：打印 AAPL 2020 MD&A 前 500 字
sample = mda_meta[(mda_meta["ticker"] == "AAPL") & (mda_meta["year"] == 2020)]
if len(sample) > 0:
    path = sample.iloc[0]["mda_path"]
    mda_text = Path(path).read_text(encoding="utf-8")
    print(f"AAPL 2020 MD&A 长度: {len(mda_text):,} chars")
    print("前 500 字：")
    print(mda_text[:500])
else:
    print("⚠️  AAPL 2020 未找到")

AAPL 2020 MD&A 长度: 31,909 chars
前 500 字：
Item 7. Management's Discussion and Analysis of Financial Condition and Results of Operations The following discussion should be read in conjunction with the consolidated financial statements and accompanying notes included in Part II, Item 8 of this Form 10-K. This section of this Form 10-K generally discusses 2020 and 2019 items and year-to-year comparisons between 2020 and 2019. Discussions of 2018 items and year-to-year comparisons between 2019 and 2018 that are not included in this Form 10-


---
## Step 4: WRDS Compustat — 拉 pilot 公司基本面

In [9]:

import wrds

db = wrds.Connection(
    wrds_username="yulinwang",
    wrds_password="Wyl20011231!"
)
print("✅ WRDS 连接成功")


Loading library list...
Done
✅ WRDS 连接成功


In [10]:
# 用 pilot 公司的 ticker 列表在 Compustat 中查 gvkey
pilot_ticker_str = "', '".join(PILOT_TICKERS)

# Step 4a: 获取 gvkey + CIK 对照
# comp.funda 里 SIC 列名是 sich（historical SIC），不是 sic
# naics 不在 funda 表里，去掉即可
company_query = f"""
    SELECT DISTINCT ON (gvkey)
        gvkey,
        tic     AS ticker,
        conm    AS company_name,
        cik,
        sich    AS sic
    FROM comp.funda
    WHERE tic IN ('{pilot_ticker_str}')
      AND indfmt  = 'INDL'
      AND datafmt = 'STD'
      AND popsrc  = 'D'
      AND consol  = 'C'
    ORDER BY gvkey, fyear DESC
"""

comp_company = db.raw_sql(company_query)
comp_company["cik"] = comp_company["cik"].astype(str).str.strip().str.zfill(10)

# 去掉重复 ticker（极少数情况：不同 gvkey 对应同一 ticker，保留最早上市的）
comp_company = comp_company.sort_values("gvkey").drop_duplicates(subset="ticker", keep="first")

print(f"Compustat 匹配到 {len(comp_company)} 家公司")
print(comp_company[["ticker", "gvkey", "cik", "sic"]].to_string(index=False))


Compustat 匹配到 20 家公司
ticker  gvkey        cik  sic
  AAPL 001690 0000320193 3663
    VZ 002136 0000732712 4812
    BA 002285 0000012927 3721
   CAT 002817 0000018230 3531
   JPM 002968 0000019617 6020
   CVX 002991 0000093410 2911
   XOM 004503 0000034088 2911
   NEE 004517 0000753308 4911
    GE 005047 0000040545 3724
    HD 005680 0000354950 5211
   JNJ 006266 0000200406 2834
   MMM 007435 0000066740 9997
   BAC 007647 0000070858 6020
   PFE 008530 0000078003 2834
     T 009899 0000732717 4812
   WMT 011259 0000104169 5331
  MSFT 012141 0000789019 7372
  AMZN 064768 0001018724 5961
    GS 114628 0000886982 6211
 GOOGL 160329 0001652044 7370


In [11]:
# Step 4b: 拉年度基本面（funda）
gvkey_str = "', '".join(comp_company["gvkey"].astype(str).tolist())

funda_query = f"""
    SELECT
        gvkey, datadate, fyear,
        tic     AS ticker,
        at      AS total_assets,
        ceq     AS book_equity,
        ni      AS net_income,
        sale    AS revenue,
        dltt    AS long_term_debt,
        csho    AS shares_out,
        prcc_f  AS price_fy_end,
        cik
    FROM comp.funda
    WHERE gvkey IN ('{gvkey_str}')
      AND fyear BETWEEN {PILOT_START - 1} AND {PILOT_END}
      AND indfmt = 'INDL'
      AND datafmt = 'STD'
      AND popsrc = 'D'
      AND consol = 'C'
    ORDER BY gvkey, fyear
"""

compustat = db.raw_sql(funda_query)
compustat["cik"] = compustat["cik"].astype(str).str.strip().str.zfill(10)

# 构建控制变量
compustat["log_assets"] = np.log(compustat["total_assets"].clip(lower=1e-6))
compustat["market_cap"] = compustat["shares_out"] * compustat["price_fy_end"]
compustat["log_mktcap"] = np.log(compustat["market_cap"].clip(lower=1e-6))
compustat["bm_ratio"]   = compustat["book_equity"] / compustat["market_cap"].clip(lower=1e-6)
compustat["roa"]        = compustat["net_income"] / compustat["total_assets"].clip(lower=1e-6)
compustat["leverage"]   = compustat["long_term_debt"] / compustat["total_assets"].clip(lower=1e-6)

compustat.to_parquet(RAW_DIR / "compustat_pilot.parquet")
print(f"✅ Compustat: {len(compustat)} 条记录")
compustat[["ticker", "fyear", "log_assets", "log_mktcap", "roa"]].head(10)

✅ Compustat: 80 条记录


,ticker,fyear,log_assets,log_mktcap,roa
0,AAPL,2017,12.835532,13.579852,0.128826
1,AAPL,2018,12.809637,13.886333,0.162775
2,AAPL,2019,12.732327,13.81065,0.16323
3,AAPL,2020,12.688153,14.491552,0.177256
4,VZ,2017,12.457388,12.282694,0.117059
5,VZ,2018,12.48684,12.355797,0.058634
6,VZ,2019,12.583574,12.444853,0.066038
7,VZ,2020,12.665018,12.40129,0.056247
8,BA,2017,11.433157,12.068552,0.088776
9,BA,2018,11.672993,12.117592,0.089128


---
## Step 5: WRDS CRSP — 拉 pilot 公司日频收益

In [12]:
# Step 5a: 获取 gvkey → permno 映射（CCM link table）
link_query = f"""
    SELECT DISTINCT
        a.gvkey,
        b.lpermno AS permno,
        b.linktype, b.linkprim,
        b.linkdt, b.linkenddt
    FROM comp.company AS a
    INNER JOIN crsp.ccmxpf_linktable AS b ON a.gvkey = b.gvkey
    WHERE a.gvkey IN ('{gvkey_str}')
      AND b.linktype IN ('LU', 'LC')
      AND b.linkprim IN ('P', 'C')
      AND b.lpermno IS NOT NULL
"""
ccm_link = db.raw_sql(link_query)
print(f"CCM link: {len(ccm_link)} 条，permno 唯一值: {ccm_link['permno'].nunique()}")
ccm_link.head()

CCM link: 29 条，permno 唯一值: 20


,gvkey,permno,linktype,linkprim,linkdt,linkenddt
0,001690,14593.0,LU,P,1980-12-12,<NA>
1,002136,65875.0,LC,P,1983-11-30,<NA>
2,002285,19561.0,LU,C,1950-01-01,1962-01-30
3,002285,19561.0,LU,P,1962-01-31,<NA>
4,002817,18542.0,LC,C,1950-01-01,1962-01-30


In [13]:
# Step 5b: 拉日频收益（多拉一年用于 estimation window）
permno_str = ", ".join(ccm_link["permno"].dropna().astype(int).astype(str).unique())

crsp_query = f"""
    SELECT a.permno, a.date, a.ret, a.retx, a.shrout, a.prc,
           b.vwretd, b.ewretd
    FROM crsp.dsf AS a
    LEFT JOIN crsp.dsi AS b ON a.date = b.date
    WHERE a.permno IN ({permno_str})
      AND a.date BETWEEN '{PILOT_START - 1}-01-01' AND '{PILOT_END + 1}-12-31'
    ORDER BY a.permno, a.date
"""

crsp = db.raw_sql(crsp_query, date_cols=["date"])
crsp["ret"] = pd.to_numeric(crsp["ret"], errors="coerce")
crsp["prc"] = crsp["prc"].abs()

crsp.to_parquet(RAW_DIR / "crsp_pilot.parquet")
print(f"✅ CRSP: {len(crsp):,} 条日频记录，{crsp['permno'].nunique()} 只股票")
crsp.head()

✅ CRSP: 25,180 条日频记录，20 只股票


,permno,date,ret,retx,shrout,prc,vwretd,ewretd
0,10107,2017-01-03,0.007081,0.007081,7730000.0,62.58,0.008205,0.009122
1,10107,2017-01-04,-0.004474,-0.004474,7730000.0,62.3,0.008618,0.013134
2,10107,2017-01-05,0.0,0.0,7730000.0,62.3,-0.000945,-0.000892
3,10107,2017-01-06,0.008668,0.008668,7730000.0,62.84,0.00231,-0.001362
4,10107,2017-01-09,-0.003183,-0.003183,7730000.0,62.64,-0.003925,-0.003245


In [14]:
db.close()
print("✅ WRDS 连接已关闭")

✅ WRDS 连接已关闭


---
## Step 6: 合并 → Master Panel

In [15]:

# ── 6.1 filings × Compustat ──────────────────────────────────────────────────
ok_statuses = ["ok", "cached", "ok_mda", "ok_fulltext_fallback"]
mda_ok = mda_meta[mda_meta["status"].isin(ok_statuses)][[
    "cik", "ticker", "accession_no", "date_filed", "year", "mda_path", "mda_len"
]].copy()
mda_ok["date_filed"] = pd.to_datetime(mda_ok["date_filed"])
mda_ok["year"] = mda_ok["year"].astype(int)

# 用 CIK 把 Compustat gvkey 接进来
comp_cik = comp_company[["gvkey", "cik", "sic"]].drop_duplicates()
panel = mda_ok.merge(comp_cik, on="cik", how="left")

# year 已经是 10-K 的 report_year/fiscal year，不再减 1
panel["fyear"] = panel["year"]

comp_slim = compustat[["gvkey", "fyear", "log_assets", "log_mktcap",
                        "bm_ratio", "roa", "leverage"]]
panel = panel.merge(comp_slim, on=["gvkey", "fyear"], how="left")

match_rate = panel["log_assets"].notna().mean() if len(panel) else float("nan")
print(f"filings × Compustat: {len(panel)} 条，基本面匹配率 {match_rate:.0%}")


filings × Compustat: 59 条，基本面匹配率 100%


In [16]:
# ── 6.2 合并 permno（考虑 link 有效期）────────────────────────────────────────
ccm = ccm_link.copy()
ccm["linkdt"]    = pd.to_datetime(ccm["linkdt"], errors="coerce")
ccm["linkenddt"] = pd.to_datetime(ccm["linkenddt"], errors="coerce").fillna(pd.Timestamp("2099-12-31"))

panel = panel.merge(ccm[["gvkey", "permno", "linkdt", "linkenddt"]], on="gvkey", how="left")

date_ok = (
    panel["permno"].isna() |
    ((panel["date_filed"] >= panel["linkdt"]) & (panel["date_filed"] <= panel["linkenddt"]))
)
panel = panel[date_ok].drop(columns=["linkdt", "linkenddt"])
panel = panel.drop_duplicates(subset=["gvkey", "fyear"], keep="first")

print(f"合并 permno 后: {len(panel)} 条，permno 匹配率 {panel['permno'].notna().mean():.0%}")

合并 permno 后: 59 条，permno 匹配率 100%


In [17]:
# ── 6.3 保存 master panel ─────────────────────────────────────────────────────
final_cols = ["gvkey", "permno", "cik", "ticker", "sic", "fyear",
              "date_filed", "accession_no", "mda_path", "mda_len",
              "log_assets", "log_mktcap", "bm_ratio", "roa", "leverage"]
final_cols = [c for c in final_cols if c in panel.columns]
master = panel[final_cols].reset_index(drop=True)

master.to_parquet(OUTPUT_DIR / "master_panel_pilot.parquet", index=False)
master.to_csv(OUTPUT_DIR / "master_panel_pilot.csv", index=False)

print("\n✅ Master Panel 保存完成")
print(master.to_string())


✅ Master Panel 保存完成
     gvkey   permno         cik ticker   sic  fyear date_filed          accession_no                                              mda_path  mda_len  log_assets  log_mktcap  bm_ratio       roa  leverage
0   160329  90319.0  0001652044  GOOGL  7370   2018 2019-02-05  0001652044-19-000004  data_pilot/filings/GOOGL/2018/000165204419000004.txt    62014   12.357901   13.496445  0.244388  0.132032  0.017234
1   160329  90319.0  0001652044  GOOGL  7370   2019 2020-02-04  0001652044-20-000008  data_pilot/filings/GOOGL/2019/000165204420000008.txt    59633   12.527826   13.734245  0.218496  0.124472  0.053525
2   160329  90319.0  0001652044  GOOGL  7370   2020 2021-02-03  0001652044-21-000010  data_pilot/filings/GOOGL/2020/000165204421000010.txt    71369   12.674876    13.98392  0.188051  0.125992  0.078463
3   001690  14593.0  0000320193   AAPL  3663   2018 2018-11-05  0000320193-18-000145   data_pilot/filings/AAPL/2018/000032019318000145.txt    57676   12.809637   13.886333

---
## Step 7: Pipeline 健康检查

In [18]:

print("="*55)
print("  Pilot Pipeline 健康检查")
print("="*55)

checks = {}

# 1. filing 覆盖率
n_expected = len(PILOT_TICKERS) * (PILOT_END - PILOT_START + 1)
ok_statuses = ["ok", "cached", "ok_mda", "ok_fulltext_fallback"]
n_ok = mda_meta["status"].isin(ok_statuses).sum() if len(mda_meta) else 0
checks["MD&A/文本下载成功率"] = f"{n_ok}/{n_expected} = {n_ok/n_expected:.0%}"

if len(master) == 0:
    checks["Master 数据"] = "0 rows ❌"
    checks["Compustat 匹配率"] = "N/A"
    checks["CRSP permno 匹配率"] = "N/A"
    checks["MD&A 中位词数"] = "N/A"
    checks["年份覆盖"] = "[]"
else:
    checks["Master 数据"] = f"{len(master):,} rows ✅"
    checks["Compustat 匹配率"] = f"{master['log_assets'].notna().mean():.0%}" if "log_assets" in master.columns else "missing log_assets ❌"
    checks["CRSP permno 匹配率"] = f"{master['permno'].notna().mean():.0%}" if "permno" in master.columns else "missing permno ❌"

    if "mda_len" in master.columns and master["mda_len"].notna().any():
        median_words = master["mda_len"].median() / 5
        checks["MD&A 中位词数"] = f"{median_words:,.0f} 词 {'✅' if 5000 < median_words < 60000 else '⚠️'}"
    else:
        checks["MD&A 中位词数"] = "N/A"

    year_col = "fyear" if "fyear" in master.columns else "year" if "year" in master.columns else None
    checks["年份覆盖"] = str(sorted(master[year_col].dropna().unique().tolist())) if year_col else "missing year/fyear ❌"

# 打印报告
for k, v in checks.items():
    print(f"  {k:25s}: {v}")

print("="*55)

all_good = (
    len(master) > 0 and
    n_ok / n_expected >= 0.8 and
    "log_assets" in master.columns and master["log_assets"].notna().mean() >= 0.8 and
    "permno" in master.columns and master["permno"].notna().mean() >= 0.8
)

if all_good:
    print("\n🎉 Pipeline 跑通！可以切换到更大样本。")
    print("   下一步：把 week1_data_collection.ipynb 里的")
    print("   target_ciks 换成 sp1500 的 CIK 列表，重跑即可。")
else:
    print("\n⚠️  部分检查未通过。优先检查缺失 ticker-year、master merge、以及短文本样本。")


  Pilot Pipeline 健康检查
  MD&A/文本下载成功率             : 59/60 = 98%
  Master 数据                : 59 rows ✅
  Compustat 匹配率            : 100%
  CRSP permno 匹配率          : 100%
  MD&A 中位词数                : 13,064 词 ✅
  年份覆盖                     : [2018, 2019, 2020]

🎉 Pipeline 跑通！可以切换到更大样本。
   下一步：把 week1_data_collection.ipynb 里的
   target_ciks 换成 sp1500 的 CIK 列表，重跑即可。


In [19]:
# 抽查：验证 CRSP 里有 AAPL 的数据（用 permno 查）
aapl_permno = master[master["ticker"] == "AAPL"]["permno"].values
if len(aapl_permno) > 0:
    sample = crsp[crsp["permno"] == aapl_permno[0]]
    print(f"AAPL permno={int(aapl_permno[0])}，CRSP 日期范围: "
          f"{sample['date'].min().date()} – {sample['date'].max().date()}，"
          f"共 {len(sample)} 个交易日")
    # 2020 filing 附近的收益
    aapl_2020_date = master[(master["ticker"]=="AAPL") & (master["fyear"]==2019)]["date_filed"]
    if len(aapl_2020_date) > 0:
        event_date = aapl_2020_date.values[0]
        window = sample[
            (sample["date"] >= pd.Timestamp(event_date) - pd.Timedelta(days=5)) &
            (sample["date"] <= pd.Timestamp(event_date) + pd.Timedelta(days=5))
        ][["date", "ret", "vwretd"]]
        print(f"\nAAPL 事件窗口（filing date ±5天）:")
        print(window.to_string(index=False))

AAPL permno=14593，CRSP 日期范围: 2017-01-03 – 2021-12-31，共 1259 个交易日

AAPL 事件窗口（filing date ±5天）:
      date       ret    vwretd
2019-10-28  0.010017  0.005234
2019-10-29 -0.023128 -0.000574
2019-10-30 -0.000123   0.00278
2019-10-31   0.02261  -0.00348
2019-11-01  0.028381  0.010212
2019-11-04  0.006567  0.003762
2019-11-05 -0.001437 -0.001073
